# Project Canopy: Kaggle training notebook 

### Load necessary packages  

In [ ]:
%pip install rasterio -q
%pip install torchgeo -q
%pip install rioxarray -q
%pip install glob2 -q
%pip install mlflow -q
%pip install dagshub -q
%pip install lightning -q

In [ ]:
import numpy as np
import torch
import lightning as L
from L.pytorch.loggers import MLFlowLogger

import torchgeo
import requests
import datetime
import os
from pathlib import Path
import dagshub

from torchgeo.datasets import RasterDataset, unbind_samples, stack_samples, IntersectionDataset
from torchgeo.samplers import RandomGeoSampler, PreChippedGeoSampler, Units
from torchgeo.transforms import indices
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp

### Link up with DagsHub to get code

In [ ]:
# Get creds
DAGSHUB_REPO_OWNER = "Omdena"
DAGSHUB_REPO_NAME = input("Enter Dagshub repo name: ") or "ProjectCanopy2"
DAGSHUB_USER_NAME = input("Enter DagsHub user name: ")
DAGSHUB_TOKEN = input("Enter DagsHub token: ")
DAGSHUB_REMOTE_PATH = input("Enter DAgshub remote path: ") or "task03_modeling"
DAGSHUB_LOCAL = input("Enter Dagshub local path: ") or "/kaggle/working/dh"

!dagshub login

In [ ]:
dh_api = dagshub.common.api.repo.RepoAPI(f"{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}")
dh_api.download(remote_path=DAGSHUB_REMOTE_PATH, local_path=DAGSHUB_LOCAL)

In [ ]:
"""
To import downloaded code, assuming os.getcwd() returns /kaggle/working:

from dh import *
"""
from dh import *
from dh.util.utils import DataPreprocessing, ModelTraining
from dh.models.lightning_models import LightningModel

In [ ]:
import importlib

In [ ]:
# TODO: run if code needs to be rewritten while working


### Set up MLFlow for experiment tracking 

In [ ]:
# import cell
mlflow_installed = !pip list -v | grep mlflow
if not mlflow_installed:
    print("Installing MLflow")
    !pip install mlflow --quiet

import mlflow

#### MLFlow configuration

Uses [these docs](https://dagshub.com/docs/integration_guide/mlflow_tracking/).

In [ ]:

MLFLOW_EXPERIMENT_NAME = input("Please enter the MLFlow experiment name or skip to use 'default'") or "default"
print("MLFlow experiment name: ",MLFLOW_EXPERIMENT_NAME)

In [ ]:
## Following code here: https://dagshub.com/docs/integration_guide/mlflow_tracking/
## Also here: https://colab.research.google.com/drive/1XLP2Ouxk-k6y9yOxc4Grp-Aq6aGcbhuj

# Point to DagsHub repo
# os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USER_NAME
# os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
os.environ['MLFLOW_TRACKING_URI'] = f'https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow'
# os.environ['MLFLOW_ARTIFACT_LOC'] = 'mlflow-artifacts:/experiments' # TODO: check this
# mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

# Create experiment and set its location
# os.environ['MLFLOW_EXPERIMENT_NAME'] = MLFLOW_EXPERIMENT_NAME
# mlflow.create_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'], os.environ['MLFLOW_ARTIFACT_LOC'])
# mlflow.set_experiment(experiment_name=os.environ['MLFLOW_EXPERIMENT_NAME'])

# set up logger for Lightning: can use different experiment_name if desired
mlf_logger = MLFlowLogger(experiment_name="lightning_logs", tracking_uri=os.environ['MLFLOW_TRACKING_URI'])

## Configure and load data

### Kaggle data paths

Once you have added a Kaggle dataset to the notebook, enter the path below. It
should start with /kaggle/input/[DATASET_NAME]/. Modify as needed for your own
directory structure.

In [ ]:
# inputs
data_base_dir = None # TODO: path to dataset root folder
data_x_dir = os.path.join(data_base_dir, None) # TODO: path to original images
data_y_dir = os.path.join(data_base_dir, None) # TODO: path to original masks

# outputs
output_base_dir = None # TODO: path to all outputs, should start /kaggle/working
reprojected_dir = os.path.join(output_base_dir, "reprojected")
reprojected_x_dir = os.path.join(reprojected_dir, "x")
reprojected_y_dir = os.path.join(reprojected_dir, "y")
reprojected_split_dir = os.path.join(reprojected_dir, "split")
objects_dir = os.path.join(output_base_dir, "objects") # model checkpoints, etc

### Datasets, dataloaders, datamodules, preprocessing

#### Reproject scenes and move images

In [ ]:

# handle imagery
orig_x_names, reproj_x_names = DataPreprocessing.get_and_copy_files(data_x_dir, 
                                                                    reprojected_x_dir)

# handle masks
orig_y_names, reproj_y_names = DataPreprocessing.get_and_copoy_files(data_y_dir,
                                                                    reprojected_y_dir)

In [ ]:
DataPreprocessing.reproject_scenes_truths(orig_y_names, 
                                          reproj_y_names, 
                                          orig_x_names, 
                                          reproj_x_names)

In [ ]:
DataPreprocessing.check(reproj_x_names, reproj_y_names)

#### Split

In [ ]:
# Split data into training and validation sets
train_images, val_images, train_masks, val_masks = DataPreprocessing.split_data(reprojected_x_dir, reprojected_y_dir, train_size = 0.7)

# Move files to train and validation directories
DataPreprocessing.move_files(reprojected_x_dir, reprojected_split_dir, 'tra_scene', train_images)
DataPreprocessing.move_files(reprojected_x_dir, reprojected_split_dir, 'val_scene', val_images)
DataPreprocessing.move_files(reprojected_y_dir, reprojected_split_dir, 'tra_truth', train_masks)
DataPreprocessing.move_files(reprojected_y_dir, reprojected_split_dir, 'val_truth', val_masks)

print("Data split and moved successfully.")

In [ ]:
# filenames
train_images = list(glob2.glob(os.path.join(reprojected_split_dir, 'tra_scene/*.tif')))
train_images.sort()
train_masks = list(glob2.glob(os.path.join(reprojected_split_dir, 'tra_truth/*.tif')))
train_masks.sort()
val_images = list(glob2.glob(os.path.join(reprojected_split_dir, 'val_scene/*.tif')))
val_images.sort()
val_masks = list(glob2.glob(os.path.join(reprojected_split_dir, 'val_scene/*.tif')))
val_masks.sort()

In [ ]:
print("Length of - \n1. Train Images = {}\n2. Train Masks = {}\n3. Validation Images = {}\n4. Validation Masks = {}".
      format(len(train_images), len(train_masks), len(val_images), len(val_masks)))
print()
print("Total size of -\nImages = {}\nMasks = {}".
      format((len(train_images) + len(val_images)), (len(train_masks) + len(val_masks))))

#### Create dataset

In [ ]:
tra_scene = os.path.join(reprojected_split_dir, 'tra_scene')
tra_truth = os.path.join(reprojected_split_dir, 'tra_truth')
val_scene = os.path.join(reprojected_split_dir, 'val_scene')
val_truth = os.path.join(reprojected_split_dir, 'val_truth')

In [ ]:
train_imgs = RasterDataset(paths=tra_scene, crs='EPSG:32636', transforms=scale, res=10)
train_msks = RasterDataset(paths=tra_truth, crs='EPSG:32636', res=10)

val_imgs = RasterDataset(paths=val_scene, crs='EPSG:32636', transforms=scale, res=10)
val_msks = RasterDataset(paths=val_truth, crs='EPSG:32636', res=10)

# IMPORTANT
train_msks.is_image = False
val_msks.is_image = False

train_sampler = RandomGeoSampler(train_imgs, size=128, length=1600)
val_sampler = RandomGeoSampler(val_imgs,size=128, length=400)

train_dset = train_imgs & train_msks
val_dset = val_imgs & val_msks

In [ ]:
train_dataloader = DataLoader(train_dset, sampler=train_sampler, batch_size=16, collate_fn=stack_samples)
valid_dataloader = DataLoader(val_dset, sampler=val_sampler, batch_size=16, collate_fn=stack_samples)

### Configure and create/load model

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# model creation
BACKBONE = 'resnet50'
segmodel = smp.Unet(BACKBONE, classes=2, activation='sigmoid').to(device)
preprocess_input = smp.encoders.get_preprocessing_fn(BACKBONE, pretrained='imagenet')

In [ ]:
for param in list(segmodel.encoder.parameters())[:]:
    param.requires_grad = False

In [ ]:
criterion = smp.losses.DiceLoss('binary')
# metrics = smp.metrics.iou_score
metric_dict = {"loss": criterion,
               "iou": smp.metrics.iou_score}

params_to_update = []
for name, param in segmodel.named_parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

optimizer = torch.optim.Adam(params=segmodel.parameters(), lr=0.001)

#### Run training and validation    

In [ ]:
epochs = 10

In [ ]:
lightning_model = LightningModel(segmodel,
                                 optimizer,
                                 metric_dict)

In [ ]:
trainer = L.Trainer(logger=mlf_logger)
trainer.fit(lightning_model,
            train_dataloader,
            valid_dataloader)

### Define MLFlow experiment method

### Conduct experiment/training run

In [ ]:
# TODO: training run code here